# Analisi e validazione del dataset `music_info`

Questo notebook documenta l’analisi esplorativa e gli interventi di pulizia effettuati sul dataset `music_info.csv`, utilizzato come base per la modellazione delle collection `tracks` e `audio_features` nel progetto MongoDB.

L’obiettivo dell’analisi è:
- comprendere la struttura del dataset;
- identificare valori null, duplicati e dati inconsistenti;
- definire una strategia per la gestione di `genre`, `tags` e delle feature audio;
- produrre una versione del dataset più coerente per l’ingestione nel database e per l’utilizzo nella web app.

## 1. Caricamento del dataset

In questa sezione viene caricato il file `music_info.csv` e viene effettuata una prima ispezione della struttura generale del dataset.

In [1]:
import numpy as np
import pandas as pd

DATA_PATH = "/kaggle/input/datasets/undefinenull/million-song-dataset-spotify-lastfm/Music Info.csv" #sostuire con il proprio path locale del dataset

music_info = pd.read_csv(DATA_PATH)
music_info.head()

,track_id,name,artist,spotify_preview_url,spotify_id,tags,genre,year,duration_ms,danceability,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,TRIOREW128F424EAF0,Mr. Brightside,The Killers,https://p.scdn.co/mp3-preview/4d26180e6961fd46...,09ZQ5TmUG8TSL56n0knqrj,"rock, alternative, indie, alternative_rock, in...",NaN,2004,222200,0.355,...,1,-4.360,1,0.0746,0.001190,0.000000,0.0971,0.240,148.114,4
1,TRRIVDJ128F429B0E8,Wonderwall,Oasis,https://p.scdn.co/mp3-preview/d012e536916c927b...,06UfBBDISthj1ZJAtX4xjj,"rock, alternative, indie, pop, alternative_roc...",NaN,2006,258613,0.409,...,2,-4.373,1,0.0336,0.000807,0.000000,0.2070,0.651,174.426,4
2,TROUVHL128F426C441,Come as You Are,Nirvana,https://p.scdn.co/mp3-preview/a1c11bb1cb231031...,0keNu0t0tqsWtExGM3nT1D,"rock, alternative, alternative_rock, 90s, grunge",RnB,1991,218920,0.508,...,4,-5.783,0,0.0400,0.000175,0.000459,0.0878,0.543,120.012,4
3,TRUEIND128F93038C4,Take Me Out,Franz Ferdinand,https://p.scdn.co/mp3-preview/399c401370438be4...,0ancVQ9wEcHVd0RrGICTE4,"rock, alternative, indie, alternative_rock, in...",NaN,2004,237026,0.279,...,9,-8.851,1,0.0371,0.000389,0.000655,0.1330,0.490,104.560,4
4,TRLNZBD128F935E4D8,Creep,Radiohead,https://p.scdn.co/mp3-preview/e7eb60e9466bc3a2...,01QoK9DA7VTeTSE3MNzp4I,"rock, alternative, indie, alternative_rock, in...",RnB,2008,238640,0.515,...,7,-9.935,1,0.0369,0.010200,0.000141,0.1290,0.104,91.841,4


## 2. Panoramica del dataset

Si analizzano dimensione, colonne e tipi logici dei dati, così da ottenere una visione iniziale delle informazioni disponibili e dei campi rilevanti per il progetto.

In [2]:
print(f"Dimensione dataset: {music_info.shape}")
print(f"Colonne dataset : {music_info.columns.tolist()}")
print("---------Info dataset------")
music_info.info()

Dimensione dataset: (50683, 21)
Colonne dataset : ['track_id', 'name', 'artist', 'spotify_preview_url', 'spotify_id', 'tags', 'genre', 'year', 'duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature']
---------Info dataset------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50683 entries, 0 to 50682
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   track_id             50683 non-null  object 
 1   name                 50683 non-null  object 
 2   artist               50683 non-null  object 
 3   spotify_preview_url  50683 non-null  object 
 4   spotify_id           50683 non-null  object 
 5   tags                 49556 non-null  object 
 6   genre                22348 non-null  object 
 7   year                 50683 non-null  int64  
 8   duration_ms          50683 non-null  int64  
 

## 3. Verifica della qualità dei dati

Questa sezione verifica la presenza di:
- valori null;
- eventuali duplicati;
- possibili anomalie che possono influenzare la modellazione MongoDB e la resa della web app.

In [3]:
# Valori null per colonna
null_counts = music_info.isnull().sum().sort_values(ascending=False)
null_perc = (music_info.isnull().mean() * 100).sort_values(ascending=False)

quality_summary = pd.DataFrame({
    "null_count": null_counts,
    "null_percentage": null_perc.round(2)
})

quality_summary

,null_count,null_percentage
genre,28335,55.91
tags,1127,2.22
artist,0,0.00
name,0,0.00
track_id,0,0.00
spotify_id,0,0.00
spotify_preview_url,0,0.00
year,0,0.00
duration_ms,0,0.00
danceability,0,0.00


In [4]:
# Duplicati globali
duplicate_rows = music_info.duplicated().sum()
print("Righe duplicate complete:", duplicate_rows)

Righe duplicate complete: 0


In [5]:
# Duplicati su track_id
duplicate_track_ids = music_info["track_id"].duplicated().sum()
print("Duplicati su track_id:", duplicate_track_ids)

Duplicati su track_id: 0


In [6]:
# Duplicati su name
duplicate_names = music_info["name"].duplicated().sum()
print("Duplicati su name:", duplicate_names)

Duplicati su name: 0


In [7]:
# Numero di valori unici per alcune colonne chiave
print("Track ID unici:", music_info["track_id"].nunique())
print("Name unici:", music_info["name"].nunique())
print("Artist unici:", music_info["artist"].nunique())

Track ID unici: 50683
Name unici: 50683
Artist unici: 8317


### Statistiche descrittive delle colonne numeriche

Per completare la verifica della qualità dei dati, vengono analizzate le statistiche descrittive delle variabili numeriche.

Questo passaggio consente di:
- individuare valori anomali o fuori scala;
- identificare feature audio potenzialmente corrotte;
- verificare la plausibilità del campo `year`.

In [8]:
# Selezione colonne numeriche rilevanti
key_numeric_cols = [
    "year",
    "duration_ms",
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "time_signature"
]

# Statistiche descrittive
music_info[key_numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
year,50683.0,2004.017323,8.860172,1900.0,2001.000000,2006.00000,2009.0000,2022.000
duration_ms,50683.0,251155.105972,107585.956031,1439.0,192733.000000,234933.00000,288193.0000,3816373.000
danceability,50683.0,0.493537,0.178838,0.0,0.364000,0.49700,0.6210,0.986
energy,50683.0,0.686486,0.251808,0.0,0.514000,0.74400,0.9050,1.000
key,50683.0,5.312748,3.568078,0.0,2.000000,5.00000,9.0000,11.000
loudness,50683.0,-8.291204,4.548365,-60.0,-10.375000,-7.20000,-5.0890,3.642
mode,50683.0,0.631060,0.482522,0.0,0.000000,1.00000,1.0000,1.000
speechiness,50683.0,0.076023,0.076007,0.0,0.035200,0.04820,0.0835,0.954
acousticness,50683.0,0.213808,0.302848,0.0,0.001400,0.03990,0.3400,0.996
instrumentalness,50683.0,0.225283,0.337049,0.0,0.000018,0.00563,0.4410,0.999


In [9]:
print("Tempo = 0:", (music_info["tempo"] == 0).sum())
print("Time_signature = 0:", (music_info["time_signature"] == 0).sum())

Tempo = 0: 10
Time_signature = 0: 10


In [10]:
print("Record con tempo e time signature = 0 -> feature audio rotte")
music_info[
    (music_info["tempo"] == 0) | (music_info["time_signature"] == 0)
][["track_id", "name", "tempo", "time_signature"]].head(20)

Record con tempo e time signature = 0 -> feature audio rotte


,track_id,name,tempo,time_signature
838,TRCTFEM12903CE53CF,Sunburn,0.0,0
1624,TRXRUTR128F4296932,Uno,0.0,0
11345,TRDPDLO128F427FD0E,Feel Good Lost,0.0,0
12081,TRQWYLY128F149F2AC,The Bluebell,0.0,0
21348,TRWRQGH12903CD60DA,Song for Someone,0.0,0
23621,TRFUMNQ128F933906A,240 Years Before Your Time,0.0,0
26227,TRLNWTE128F14942B2,[untitled],0.0,0
40029,TRCYKDU128F92F1856,Xylem,0.0,0
40473,TRQKNOS128F9310736,Local Boy Makes God,0.0,0
42732,TRDZLJH128F428CB73,Cheerleader Corpses,0.0,0


In [11]:
# Record con year invalido, ovvero 1900
invalid_years = music_info[music_info["year"] == 1900]

print(f"Record con year = 1900: {invalid_years.shape[0]} -> data inesatta")
invalid_years[["track_id", "name", "artist", "year"]]

Record con year = 1900: 8 -> data inesatta


,track_id,name,artist,year
7253,TRCQKHP128EF35424B,Street Life,Randy Crawford,1900
20981,TRVFYPS12903CE6989,Clean Up Woman,Betty Wright,1900
28939,TRLIHGJ128F93466DD,Inside of Me,Benny Benassi,1900
40766,TRUHYRG128F4270FCA,Pet Grief,The Radio Dept.,1900
40773,TRUPZCY128F4270FCC,I Wanted You To Feel The Same,The Radio Dept.,1900
40854,TRXPBOK128F4263F73,Do What You Wanna Do,Acid House Kings,1900
41247,TRRTDHQ12903CB63A2,Let Me Have This,The Radio Dept.,1900
47618,TRSFPNL12903CB64E7,What You Sell,The Radio Dept.,1900


### Definizione del flag `has_audio_features`

Dai controlli effettuati emerge la presenza di record con feature audio non valide, in particolare nei casi in cui:
- `tempo = 0`
- `time_signature = 0`

Per distinguere i record affidabili da quelli corrotti, viene introdotto il campo booleano `has_audio_features`, impostato a `True` solo quando entrambe le feature sono valide.

In [12]:
music_info["has_audio_features"] = (
    (music_info["tempo"] != 0) & (music_info["time_signature"] != 0)
)

music_info["has_audio_features"].value_counts()

has_audio_features
True     50673
False       10
Name: count, dtype: int64

### Correzione delle date errate

I record con `year = 1900` sono considerati errati.  
Dato il numero molto limitato di casi (8), si procede con una correzione manuale.

Le correzioni sono effettuate sulla base di fonti affidabili trovate cercando su Internet e vengono applicate direttamente al dataset.

In [13]:
# Correzioni manuali: year corretto
year_corrections = {
    7253: 2013,   # Street Life
    20981: 2013,  # Clean Up Woman
    28939: 2003,  # Inside Of Me
    40766: 2006,  # Pet Grief
    40773: 2006,  # I Wanted You To Feel The Same
    40854: 2005,  # Do What You Wanna Do
    41247: 2019,  # Let Me Have This
    47618: 2011   # What You Sell
}

for idx, new_year in year_corrections.items():
    music_info.loc[idx, "year"] = new_year

print("Record con year = 1900 dopo correzione:",
      (music_info["year"] == 1900).sum())


Record con year = 1900 dopo correzione: 0


## 4. Analisi del campo `tags`

Il campo `tags` contiene etichette descrittive associate ai brani, spesso multiple e separate da virgola.

L’obiettivo di questa analisi è:
- valutare la presenza di valori null;
- analizzare la distribuzione dei tag;
- determinare il ruolo del campo all’interno della web app.

In [14]:
# Valori null in tags
tags_null = music_info["tags"].isnull().sum()
tags_total = len(music_info)

print(f"Tags null: {tags_null} ({(tags_null/tags_total)*100:.2f}%)")

Tags null: 1127 (2.22%)


### Osservazione

Il campo `tags` presenta una percentuale molto ridotta di valori mancanti, rendendolo complessivamente affidabile come informazione aggiuntiva.

In [15]:
# Analisi distribuzione tags
tag_counts = (
    music_info[music_info["tags"].notnull()]["tags"]
    .str.split(", ")
    .explode()
    .value_counts()
)

tag_counts.head(25)

tags
rock                 10684
indie                 7287
electronic            6594
alternative           6274
pop                   4651
female_vocalists      4517
alternative_rock      4137
indie_rock            3801
metal                 3181
classic_rock          2779
singer_songwriter     2747
experimental          2678
chillout              2655
ambient               2561
folk                  2530
punk                  2501
british               2410
instrumental          2354
hard_rock             2339
80s                   2307
dance                 2197
90s                   2165
acoustic              2043
hip_hop               1993
death_metal           1938
Name: count, dtype: int64

### Osservazione

I tag più frequenti includono generi e descrittori comuni come rock, indie, electronic e pop.

Questo indica che il campo `tags` contiene informazioni utili ma ridondanti rispetto a `genre`, risultando più adatto come metadato descrittivo aggiuntivo.

### Decisione progettuale

Il campo `tags` verrà mantenuto come informazione opzionale nel sistema.

In particolare:
- non verrà utilizzato come label principale;
- sarà mostrato nella web app solo se disponibile;
- in assenza di dati, verrà omesso oppure sostituito con un messaggio come `No tags available`.

Questa scelta consente di arricchire l’esperienza utente senza compromettere la consistenza del sistema nei casi in cui i tag non sono presenti.

## 5. Analisi e gestione del campo `genre`

Il campo `genre` rappresenta la principale etichetta semantica del brano ed è centrale per:
- la visualizzazione nella web app;
- le funzionalità di filtraggio e ricerca.

Per questo motivo è fondamentale garantirne completezza e coerenza.

In [16]:
genre_null = music_info["genre"].isnull().sum()
genre_total = len(music_info)

print(f"Genre null: {genre_null} ({(genre_null/genre_total)*100:.2f}%)")

Genre null: 28335 (55.91%)


### Osservazione

Il campo `genre` presenta una percentuale molto elevata di valori mancanti (circa il 55%).

Questa situazione non è compatibile con l’utilizzo previsto nella web app, dove ogni brano deve avere un genere associato.

In [17]:
genre_counts = (
    music_info[music_info["genre"].notnull()]["genre"]
    .value_counts()
)

genre_counts.head(15)

genre
Rock          9965
Electronic    3710
Metal         2516
Pop           1145
Rap            821
Jazz           793
RnB            696
Reggae         691
Country        607
Punk           383
Folk           355
New Age        237
Blues          189
World          140
Latin          100
Name: count, dtype: int64

### Osservazione

I generi più frequenti includono Rock, Electronic, Metal e Pop.

La distribuzione è sbilanciata, con alcune classi molto più rappresentate di altre, aspetto da considerare nella fase di classificazione.

### Decisione progettuale

Il campo `genre` viene trattato come **label principale** del brano.

Per garantire un’esperienza utente uniforme, si decide di:
- eliminare i valori null;
- completare il dataset tramite imputazione supervisionata.

A differenza di `tags`, `genre` deve essere sempre presente.

In [18]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# split dataset
df_train = music_info[music_info['genre'].notnull()]
df_pred = music_info[music_info['genre'].isnull()].copy()

# encoding target
le = LabelEncoder()
y = le.fit_transform(df_train['genre'])

# features selection
features = [
    'danceability', 'energy', 'loudness',
    'speechiness', 'acousticness',
    'instrumentalness', 'liveness',
    'valence', 'tempo'
]

X = df_train[features]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

model_rf = RandomForestClassifier(random_state=42)
model_rf.fit(X_train, y_train)

# evaluation
y_pred_test_rf = model_rf.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_test_rf)
print("Random Forest Accuracy:", accuracy_rf)

# prediction sui missing
X_pred = df_pred[features]
y_pred_rf = model_rf.predict(X_pred)

df_pred['genre_pred_rf'] = le.inverse_transform(y_pred_rf)

Random Forest Accuracy: 0.5263982102908278


In [20]:
print("=== RANDOM FOREST REPORT===")
print(classification_report(
    y_test,
    y_pred_test_rf,
    target_names=le.classes_,
    zero_division=0
))

=== RANDOM FOREST REPORT===
              precision    recall  f1-score   support

       Blues       0.00      0.00      0.00        36
     Country       0.15      0.02      0.03       114
  Electronic       0.55      0.52      0.54       700
        Folk       0.00      0.00      0.00        70
        Jazz       0.30      0.02      0.03       184
       Latin       0.00      0.00      0.00        20
       Metal       0.55      0.41      0.47       530
     New Age       1.00      0.02      0.04        53
         Pop       0.27      0.03      0.05       219
        Punk       0.00      0.00      0.00        92
         Rap       0.56      0.44      0.50       174
      Reggae       0.35      0.10      0.16       147
         RnB       0.23      0.02      0.04       146
        Rock       0.53      0.85      0.65      1962
       World       0.00      0.00      0.00        23

    accuracy                           0.53      4470
   macro avg       0.30      0.16      0.17      447

In [21]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


model_lr = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        class_weight='balanced',
        max_iter=5000,
        random_state=42
    )
)

model_lr.fit(X_train, y_train)

# evaluation
y_pred_test_lr = model_lr.predict(X_test)
accuracy_lr = accuracy_score(y_test, y_pred_test_lr)
print("Logistic Regression Accuracy:", accuracy_lr)

# prediction sui missing
y_pred_lr = model_lr.predict(X_pred)

df_pred['genre_pred_lr'] = le.inverse_transform(y_pred_lr)

Logistic Regression Accuracy: 0.2483221476510067


In [22]:
print("=== LOGISTIC REGRESSION REPORT ===")
print(classification_report(
    y_test,
    y_pred_test_lr,
    target_names=le.classes_
))

=== LOGISTIC REGRESSION REPORT ===
              precision    recall  f1-score   support

       Blues       0.00      0.00      0.00        36
     Country       0.10      0.14      0.11       114
  Electronic       0.57      0.44      0.50       700
        Folk       0.08      0.31      0.13        70
        Jazz       0.00      0.00      0.00       184
       Latin       0.04      0.40      0.07        20
       Metal       0.39      0.58      0.46       530
     New Age       0.08      0.45      0.14        53
         Pop       0.11      0.17      0.13       219
        Punk       0.05      0.39      0.09        92
         Rap       0.34      0.51      0.41       174
      Reggae       0.12      0.30      0.17       147
         RnB       0.12      0.06      0.08       146
        Rock       0.70      0.10      0.18      1962
       World       0.04      0.26      0.06        23

    accuracy                           0.25      4470
   macro avg       0.18      0.27      0.17  

In [23]:
print("RF:", accuracy_rf)
print("LR:", accuracy_lr)

RF: 0.5263982102908278
LR: 0.2483221476510067


In [24]:
df_pred['genre_pred_rf'].value_counts(normalize=True)

genre_pred_rf
Rock          0.700547
Electronic    0.135486
Metal         0.110429
Rap           0.025022
Reggae        0.009741
Pop           0.005964
Jazz          0.004976
RnB           0.002470
Country       0.002365
Folk          0.001588
New Age       0.000953
Blues         0.000282
Punk          0.000106
World         0.000071
Name: proportion, dtype: float64

In [25]:
df_pred['genre_pred_lr'].value_counts(normalize=True)

genre_pred_lr
Metal         0.190118
Punk          0.129310
Electronic    0.097194
Reggae        0.084313
Folk          0.079372
Pop           0.077360
New Age       0.071925
Rock          0.059432
Latin         0.056150
Rap           0.050856
World         0.040268
Country       0.038998
RnB           0.020046
Jazz          0.003564
Blues         0.001094
Name: proportion, dtype: float64

In [26]:
df_pred[['genre_pred_rf', 'genre_pred_lr']].head(20)

,genre_pred_rf,genre_pred_lr
0,Rock,Metal
1,Rock,Rock
3,Rock,Rock
5,Rock,Punk
6,Rock,Punk
7,Rock,Rock
9,Rock,Country
11,Electronic,Electronic
14,Rock,Rock
15,Rock,Pop


### Confronto modelli

Random Forest mostra prestazioni significativamente migliori rispetto a Logistic Regression.

Questo è coerente con:
- la natura non lineare dei dati;
- la presenza di feature audio complesse.

Si sceglie quindi Random Forest come modello finale.

In [27]:


music_info['genre_source'] = music_info['genre'].apply(
    lambda x: 'original' if pd.notnull(x) else 'predicted_rf'
)

music_info.loc[
    music_info['genre'].isnull(),
    'genre'
] = df_pred['genre_pred_rf'].values

print("Genre null rimasti:", music_info['genre'].isnull().sum())
print(music_info['genre_source'].value_counts())



Genre null rimasti: 0
genre_source
predicted_rf    28335
original        22348
Name: count, dtype: int64


In [28]:
music_info.head(20)

,track_id,name,artist,spotify_preview_url,spotify_id,tags,genre,year,duration_ms,danceability,...,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,has_audio_features,genre_source
0,TRIOREW128F424EAF0,Mr. Brightside,The Killers,https://p.scdn.co/mp3-preview/4d26180e6961fd46...,09ZQ5TmUG8TSL56n0knqrj,"rock, alternative, indie, alternative_rock, in...",Rock,2004,222200,0.355,...,1,0.0746,0.001190,0.000000,0.0971,0.240,148.114,4,True,predicted_rf
1,TRRIVDJ128F429B0E8,Wonderwall,Oasis,https://p.scdn.co/mp3-preview/d012e536916c927b...,06UfBBDISthj1ZJAtX4xjj,"rock, alternative, indie, pop, alternative_roc...",Rock,2006,258613,0.409,...,1,0.0336,0.000807,0.000000,0.2070,0.651,174.426,4,True,predicted_rf
2,TROUVHL128F426C441,Come as You Are,Nirvana,https://p.scdn.co/mp3-preview/a1c11bb1cb231031...,0keNu0t0tqsWtExGM3nT1D,"rock, alternative, alternative_rock, 90s, grunge",RnB,1991,218920,0.508,...,0,0.0400,0.000175,0.000459,0.0878,0.543,120.012,4,True,original
3,TRUEIND128F93038C4,Take Me Out,Franz Ferdinand,https://p.scdn.co/mp3-preview/399c401370438be4...,0ancVQ9wEcHVd0RrGICTE4,"rock, alternative, indie, alternative_rock, in...",Rock,2004,237026,0.279,...,1,0.0371,0.000389,0.000655,0.1330,0.490,104.560,4,True,predicted_rf
4,TRLNZBD128F935E4D8,Creep,Radiohead,https://p.scdn.co/mp3-preview/e7eb60e9466bc3a2...,01QoK9DA7VTeTSE3MNzp4I,"rock, alternative, indie, alternative_rock, in...",RnB,2008,238640,0.515,...,1,0.0369,0.010200,0.000141,0.1290,0.104,91.841,4,True,original
5,TRUMISQ128F9340BEE,Somebody Told Me,The Killers,https://p.scdn.co/mp3-preview/0d07673cfb46218a...,0FNmIQ7u45Lhdn6RHhSLix,"rock, alternative, indie, pop, alternative_roc...",Rock,2005,198480,0.508,...,0,0.0847,0.000087,0.000643,0.0641,0.704,138.030,4,True,predicted_rf
6,TRVCCWR128F9304A30,Viva la Vida,Coldplay,https://p.scdn.co/mp3-preview/ab747fed1bfab2ac...,08A1lZeyLMWH58DT6aYjnC,"rock, alternative, indie, pop, alternative_roc...",Rock,2013,235384,0.588,...,1,0.1050,0.153000,0.000000,0.0634,0.520,137.973,4,True,predicted_rf
7,TRXOGZT128F424AD74,Karma Police,Radiohead,https://p.scdn.co/mp3-preview/5a09f5390e2862af...,01puceOqImrzSfKDAcd1Ia,"rock, alternative, indie, alternative_rock, in...",Rock,1996,264066,0.360,...,1,0.0260,0.062600,0.000092,0.1720,0.317,74.807,4,True,predicted_rf
8,TRMZXEW128F9341FD5,The Scientist,Coldplay,https://p.scdn.co/mp3-preview/95cb9df1b056d759...,0GSSsT9szp0rJkBrYkzy6s,"rock, alternative, indie, pop, alternative_roc...",Rock,2007,311014,0.566,...,1,0.0242,0.715000,0.000014,0.1200,0.173,146.365,4,True,original
9,TRUJIIV12903CA8848,Clocks,Coldplay,https://p.scdn.co/mp3-preview/24c7fe858b234e3c...,0BCPKOYdS2jbQ8iyB56Zns,"rock, alternative, indie, pop, alternative_roc...",Rock,2002,307879,0.577,...,0,0.0279,0.599000,0.011500,0.1830,0.255,130.970,4,True,predicted_rf


In [29]:
music_info.to_csv("/kaggle/working/music_info_cleaned.csv", index=False)  #sostuire path con il proprio per il salvataggio

### Risultato finale

Il campo `genre` è stato completamente riempito, eliminando tutti i valori null.

L’introduzione del campo `genre_source` consente di distinguere tra:
- valori originali;
- valori predetti tramite modello.

Questo approccio garantisce:
- coerenza del dataset;
- uniformità nella web app;
- tracciabilità delle modifiche effettuate.

## 6. Conclusioni finali

### Sintesi dell’analisi

L’analisi del dataset `music_info` ha permesso di comprendere struttura, qualità e criticità dei dati, evidenziando diversi aspetti rilevanti per la progettazione del sistema.

In particolare:

- il dataset contiene informazioni sui brani, incluse feature audio, metadati e campi descrittivi (`genre`, `tags`);
- il campo `tags` risulta quasi completo e utile come informazione opzionale;
- il campo `genre` presenta una percentuale molto elevata di valori mancanti (~55%);
- sono presenti valori inconsistenti nelle feature audio (`tempo = 0`, `time_signature = 0`);
- sono stati individuati pochi casi di date errate (`year = 1900`), corretti manualmente.

---

### Interventi effettuati

Per migliorare la qualità del dataset sono state adottate le seguenti strategie:

- completamento del campo `genre` tramite classificazione supervisionata (Random Forest);
- introduzione del campo booleano `has_audio_features` per identificare record con feature audio valide;
- correzione manuale delle date errate, data la loro bassa numerosità;
- mantenimento del campo `tags` come metadato opzionale.

---

### Implicazioni per il modello MongoDB

Le decisioni prese influenzano direttamente la progettazione delle collection:

- `genre` sarà un campo sempre presente nella collection `tracks`;
- `tags` sarà opzionale e utilizzato solo per arricchire la visualizzazione;
- le feature audio saranno gestite separatamente e validate tramite `has_audio_features`;
- il dataset risultante è coerente e pronto per la fase di ingestione in MongoDB.

---

### Impatto sulla web app

Le scelte effettuate garantiscono:

- uniformità nella visualizzazione dei brani (ogni traccia ha un genere);
- maggiore robustezza nei dati mostrati all’utente;
- gestione elegante dei campi opzionali (come `tags`);
- possibilità di implementare filtri e query affidabili basati su `genre` e feature audio.

---

### Considerazioni finali

Il processo di analisi e pulizia ha trasformato un dataset con diverse criticità in una base dati coerente, utilizzabile sia per la modellazione NoSQL sia per l’integrazione con la web app.

Questa fase rappresenta un passaggio fondamentale per garantire la qualità complessiva del sistema nelle fasi successive del progetto.